# Guía del docente — KPIs de monitoreo CEFIT

Referencia para enseñar los 9 indicadores del dashboard de Grafana.  
**Nivel:** desarrolladores junior que nunca han trabajado con observabilidad.

---

## Mapa del dashboard

```
┌──────────────────┬──────────────────┬──────────────────┐
│  Request Rate    │   Error Rate     │  Latencia p95    │  ← fila de estado
└──────────────────┴──────────────────┴──────────────────┘
┌─────────────────────────────┬────────────────────────────┐
│  Request Rate por Status    │  Latencia (p50/p95/p99)    │  ← tendencias
└─────────────────────────────┴────────────────────────────┘
┌────────────────┬────────────┬──────────────┬─────────────┐
│  Heap Memory   │  CPU Usage │ Event Loop   │   Handles   │  ← recursos
└────────────────┴────────────┴──────────────┴─────────────┘
```

---
## KPI 1 — Request Rate

**¿Qué mide?** Cuántas peticiones HTTP llegan al servidor por segundo, promediado en los últimos 5 minutos.

**Expresión PromQL**
```promql
sum(rate(cefit_http_requests_total[5m]))
```

**Objetivos de aprendizaje**
- Entender qué es una ventana deslizante de tiempo (`[5m]`)
- Distinguir `rate()` (por segundo) de `increase()` (total acumulado)
- Relacionar picos de tráfico con eventos reales (campañas, horario pico)

**Umbrales orientativos**

| Tráfico | Valor aprox. |
|---|---|
| Sistema en reposo | < 0.1 req/s |
| Uso normal | 1 – 10 req/s |
| Carga alta | > 50 req/s |

> **Pregunta para discusión:** *"Si el Request Rate sube a 200 req/s a las 3 AM, ¿qué puede estar pasando?"*

In [ ]:
# Demostración: simular rate() de Prometheus
# Prometheus calcula cuántos eventos ocurrieron en una ventana y los divide por los segundos

peticiones_en_5_min = 1500
ventana_segundos = 5 * 60  # 300 s

request_rate = peticiones_en_5_min / ventana_segundos
print(f"Request Rate = {peticiones_en_5_min} / {ventana_segundos} = {request_rate:.2f} req/s")

---
## KPI 2 — Error Rate

**¿Qué mide?** Porcentaje de respuestas HTTP 5xx sobre el total de peticiones.

**Expresión PromQL**
```promql
sum(rate(cefit_http_requests_total{status_code=~"5.."}[5m]))
  / sum(rate(cefit_http_requests_total[5m])) * 100
```

**Objetivos de aprendizaje**
- Diferencia entre errores 4xx (cliente) y 5xx (servidor)
- El concepto de SLO: p.ej. "error rate < 1%"
- Por qué un Error Rate de 0% con Request Rate de 0 no significa nada

**Umbrales**

| Estado | Valor |
|---|---|
| Saludable | < 1 % |
| Advertencia | 1 – 5 % |
| Crítico | > 5 % |

> **Pregunta para discusión:** *"¿Un Error Rate de 0% siempre significa que todo está bien?"*

In [ ]:
# Ejercicio para clase: calcular el Error Rate
# Cambiar estos valores y pedir al estudiante que prediga el resultado

total_peticiones = 1000
peticiones_5xx   = 12

error_rate = (peticiones_5xx / total_peticiones) * 100
print(f"Error Rate = ({peticiones_5xx} / {total_peticiones}) x 100 = {error_rate:.1f}%")

if error_rate < 1:
    print("Estado: SALUDABLE")
elif error_rate < 5:
    print("Estado: ADVERTENCIA")
else:
    print("Estado: CRÍTICO")

---
## KPI 3 — Latencia p95

**¿Qué mide?** El tiempo de respuesta que el 95% de las peticiones no supera.

**Expresión PromQL**
```promql
histogram_quantile(
  0.95,
  sum by(le) (rate(cefit_http_request_duration_seconds_bucket[5m]))
)
```

**Objetivos de aprendizaje**
- Qué es un percentil y por qué se usa p95 en vez del promedio
- Por qué el promedio oculta outliers
- Diferencia entre p50, p95 y p99

**Umbrales**

| Estado | Valor |
|---|---|
| Excelente | < 100 ms |
| Aceptable | 100 – 500 ms |
| Degradado | > 500 ms |

> **Ejercicio:** Pedir al grupo que calcule el p95 a mano con 20 tiempos de respuesta inventados.

In [ ]:
import math

tiempos_ms = [8, 10, 12, 15, 18, 20, 22, 25, 28, 30,
              35, 40, 45, 50, 60, 70, 80, 95, 200, 950]

tiempos_ms.sort()
n = len(tiempos_ms)

def percentil(datos, k):
    pos = math.ceil(k / 100 * len(datos))
    return datos[pos - 1]

p50 = percentil(tiempos_ms, 50)
p95 = percentil(tiempos_ms, 95)
p99 = percentil(tiempos_ms, 99)
promedio = sum(tiempos_ms) / n

print(f"Promedio : {promedio:.1f} ms  ← oculta los outliers")
print(f"p50      : {p50} ms")
print(f"p95      : {p95} ms  ← 95% de usuarios tardaron esto o menos")
print(f"p99      : {p99} ms  ← peor caso (1 de cada 100)")

---
## KPI 4 — Request Rate por Status

**¿Qué mide?** Request Rate desglosado por código HTTP a lo largo del tiempo.

**Expresión PromQL**
```promql
sum by(status_code) (rate(cefit_http_requests_total[5m]))
```

**Objetivos de aprendizaje**
- Leer una serie temporal con múltiples líneas (una por código HTTP)
- Reconocer patrones: pico de 401 = posible ataque de fuerza bruta
- Cómo usar etiquetas como dimensión de filtro en PromQL

> **Pregunta para discusión:** *"¿Cuál es la diferencia entre ver 401 en tus logs y ver un pico de 401 en Grafana?"*

---
## KPI 5 — Latencia: p50 / p95 / p99 juntos

**¿Qué mide?** La distribución completa de tiempos de respuesta usando los tres percentiles a la vez.

**Expresiones PromQL**
```promql
histogram_quantile(0.50, ...)   -- mediana
histogram_quantile(0.95, ...)   -- 95% de usuarios
histogram_quantile(0.99, ...)   -- peor caso
```

**Objetivos de aprendizaje**
- La diferencia entre p50 y p99 revela la "cola larga" de latencia
- Un p99 muy alto con p50 bajo = problema solo con ciertas rutas
- Por qué los SLA usan p99 y no promedio

In [ ]:
# Visualización de percentiles en consola para clase
import random
random.seed(42)

# Simular tiempos de respuesta con distribución realista
tiempos = sorted([random.randint(5, 80) for _ in range(95)] +
                 [random.randint(200, 2000) for _ in range(5)])  # 5% lentos

def pct(datos, k):
    return datos[math.ceil(k / 100 * len(datos)) - 1]

print(f"Total muestras : {len(tiempos)}")
print(f"p50  : {pct(tiempos, 50):>6} ms  (usuario típico)")
print(f"p95  : {pct(tiempos, 95):>6} ms  (1 de cada 20 tardó más)")
print(f"p99  : {pct(tiempos, 99):>6} ms  (1 de cada 100 tardó más)")
print(f"Max  : {max(tiempos):>6} ms")
print(f"Media: {sum(tiempos)/len(tiempos):>6.1f} ms  (el promedio engaña)")

---
## KPI 6 — Heap Memory

**¿Qué mide?** Memoria RAM usada vs. total reservada por el proceso Node.js.

**Expresiones PromQL**
```promql
cefit_nodejs_heap_size_used_bytes
cefit_nodejs_heap_size_total_bytes
```

**Objetivos de aprendizaje**
- Qué es el heap en un proceso Node.js
- Señales de memory leak: la línea "usada" sube sin bajar nunca
- Relación con el límite `memory: 256m` en `docker-compose.yml`

| Estado | Condición |
|---|---|
| Normal | usado < 70% del total |
| Advertencia | usado > 70% y subiendo |
| Crítico | usado ≈ total |

In [ ]:
# Simular lectura de métricas de memoria
heap_usada_mb  = 95
heap_total_mb  = 150
limite_docker_mb = 256

uso_pct = (heap_usada_mb / heap_total_mb) * 100
margen_docker_pct = ((limite_docker_mb - heap_usada_mb) / limite_docker_mb) * 100

print(f"Heap usada  : {heap_usada_mb} MB")
print(f"Heap total  : {heap_total_mb} MB")
print(f"Uso heap    : {uso_pct:.1f}%")
print(f"Límite Docker: {limite_docker_mb} MB → margen restante: {margen_docker_pct:.1f}%")

if uso_pct > 90:
    print("⚠️  CRÍTICO: el proceso puede caerse")
elif uso_pct > 70:
    print("⚠️  ADVERTENCIA: investigar memory leak")
else:
    print("✓ NORMAL")

---
## KPI 7 — CPU Usage

**¿Qué mide?** Porcentaje de CPU del proceso Node.js en el último minuto.

**Expresión PromQL**
```promql
rate(cefit_process_cpu_seconds_total[1m]) * 100
```

**Objetivos de aprendizaje**
- Por qué Node.js puede tener CPU alta siendo de un solo hilo
- Correlacionar picos de CPU con picos de Request Rate
- Diferencia entre CPU user y CPU system

| Valor | Interpretación |
|---|---|
| < 30 % | Normal |
| 30 – 70 % | Carga moderada |
| > 70 % sostenido | Código bloqueante |

---
## KPI 8 — Event Loop Lag

**¿Qué mide?** Retraso en el bucle de eventos de Node.js.

**Expresión PromQL**
```promql
cefit_nodejs_eventloop_lag_seconds
```

**Objetivos de aprendizaje**
- Por qué Node.js es de un solo hilo y qué implica eso
- Qué operaciones bloquean el event loop
- Relación directa con la latencia del usuario

| Estado | Valor |
|---|---|
| Saludable | < 10 ms |
| Advertencia | 10 – 100 ms |
| Crítico | > 100 ms |

> **Pregunta para discusión:** *"bcrypt tiene un costo de 12 en este proyecto. ¿Podría eso aumentar el event loop lag?"*  
> (Sí — aunque `bcrypt.hash()` es async, la parte de CPU es pesada)

---
## KPI 9 — Active Handles / Requests

**¿Qué mide?** Conexiones y peticiones activas en el proceso Node.js.

**Expresiones PromQL**
```promql
cefit_nodejs_active_handles_total
cefit_nodejs_active_requests_total
```

**Objetivos de aprendizaje**
- Qué es un handle en libuv (capa I/O de Node)
- Señal de leak de conexiones: handles sube sin bajar nunca
- Por qué `active_handles` alto con CPU baja indica espera de BD (I/O-bound)

---
## Secuencia de diagnóstico para clase

```
¿Hay alertas?
    │
    ├─ Error Rate > 1% ──────────→ revisar logs de la app
    │
    ├─ Latencia p95 > 500ms
    │       ├─ CPU > 70%?  ────→ código bloqueante / event loop lag
    │       └─ Handles alto? ──→ BD lenta o pool de conexiones agotado
    │
    └─ Request Rate = 0 ─────────→ Caddy caído o problema de red
```

---
## Evaluación propuesta

1. Levantar el stack con `docker compose --env-file .env.local up -d`
2. Generar carga:
   ```bash
   for i in $(seq 50); do curl -sk https://localhost/api/health & done
   ```
3. Abrir Grafana en `https://localhost:3001`
4. Pedir al estudiante que identifique y explique cada KPI en vivo
5. Generar errores a propósito y observar el Error Rate